# Horizon-area evolution in ΛCDM+S

Interactive demo: how the sigmoid transition parameters $(k, t_{\rm crit})$ change the Hubble-horizon area history $A_H \propto 1/H^2$ relative to flat ΛCDM sharing the same $(H_0, \Omega_{\Lambda+S})$.

Because the entropy sector is live (post-audit pipeline), different $(k, t_{\rm crit})$ give **different** curves — run the cell and vary them.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
import numpy as np
import matplotlib.pyplot as plt
import lcdm_plus_s.bayesian_validation as bv

z = np.linspace(0, 2, 80)
H0, OmL = 73.0, 0.688

fig, ax = plt.subplots(figsize=(8, 5))
for k, tc in [(0.10, 15.0), (0.372, 15.827), (0.45, 17.0)]:
    bg = bv.BackgroundParams(H0_kms_mpc=H0, Omega_Lambda=OmL, k_gyr=k, t_crit_gyr=tc)
    sol = bv.solve_background(bg, nsteps=400, normalization_iterations=2)
    ref = bv.BackgroundParams(H0_kms_mpc=H0, Omega_Lambda=OmL, lcdm_limit=True)
    ratio = [(1/max(sol.hubble_of_z(zz),1e-30)**2)/(1/max(bv.lcdm_hubble(ref, zz),1e-30)**2) for zz in z]
    ax.plot(z, ratio, label=f'k={k}, t_crit={tc} Gyr')
ax.axhline(1, color='k', lw=1, label='flat ΛCDM')
ax.set_xlabel('z'); ax.set_ylabel(r'$A_H^{+S}/A_H^{\Lambda CDM}$')
ax.set_title('Entropy-sector imprint on the horizon area')
ax.legend(); plt.show()

Entropy monotonicity check: $S_H \propto 1/H^2$ must not decrease while the model claims GSL compliance. Inspect $\dot S_H$ along a posterior-median trajectory.

In [ ]:
bg = bv.BackgroundParams(H0_kms_mpc=73.0, Omega_Lambda=0.688, k_gyr=0.372, t_crit_gyr=15.827)
sol = bv.solve_background(bg, nsteps=600, normalization_iterations=2)
t = np.asarray(sol.t_gyr); H = np.asarray(sol.H)
S = 1.0 / H**2  # ∝ horizon entropy
dS = np.gradient(S, t)
plt.figure(figsize=(8,4))
plt.plot(t, dS)
plt.axhline(0, color='k', lw=1)
plt.xlabel('t [Gyr]'); plt.ylabel(r'$\dot S_H$ (arb.)')
plt.title('GSL check: is horizon entropy monotonic along the fitted history?')
plt.show()
print('min dS/dt =', dS.min(), '(negative values violate the stated GSL postulate)')